In [1]:
!pip install transformers[torch] datasets sacremoses

In [3]:
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score

# --- 1. เตรียมข้อมูล ---
print("🔄 กำลังโหลดและสุ่มข้อมูล 1,000 แถว...")
df = pd.read_csv('phishing_email.csv').dropna().drop_duplicates()
df_th = df.sample(n=1000, random_state=42).reset_index(drop=True)

# --- 2. แปลภาษาแบบ Batch Processing (Manual Mode - No Pipeline) ---
print("🚀 กำลังโหลดโมเดล NLLB-200...")
model_name_trans = "facebook/nllb-200-distilled-600M"
trans_tokenizer = AutoTokenizer.from_pretrained(model_name_trans)
trans_model = AutoModelForSeq2SeqLM.from_pretrained(model_name_trans).to("cuda")

print("⏳ เริ่มแปลภาษา 1,000 แถวแบบ Batch... (คาดว่าใช้เวลา 10-15 นาที)")

texts_to_translate = df_th['text_combined'].apply(lambda x: str(x)[:400]).tolist()
translated_results = []
batch_size = 16 # ปรับความเร็วตรงนี้

for i in range(0, len(texts_to_translate), batch_size):
    batch = texts_to_translate[i : i + batch_size]

    # Tokenize ทั้ง Batch พร้อมกัน
    inputs = trans_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=400).to("cuda")

    # Generate ผลลัพธ์ทั้ง Batch
    with torch.no_grad():
        translated_tokens = trans_model.generate(
            **inputs,
            forced_bos_token_id=trans_tokenizer.convert_tokens_to_ids("tha_Thai"),
            max_length=400
        )

    # Decode กลับเป็นข้อความ
    batch_res = trans_tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)
    translated_results.extend(batch_res)

    if (i + batch_size) % 160 == 0 or (i + batch_size) >= 1000:
        processed = min(i + batch_size, 1000)
        print(f"✅ แปลเสร็จแล้ว {processed}/1000 แถว...")

df_th['text_thai'] = translated_results

# --- 3. ตรวจสอบข้อมูลหลังแปล ---
df_th = df_th[df_th['text_thai'].str.strip() != ""].reset_index(drop=True)
print(f"📊 สรุปข้อมูลที่แปลสำเร็จ: {len(df_th)} แถว")

# --- 4. เตรียม Dataset สำหรับ Multilingual BERT ---
if len(df_th) > 100:
    print("🔥 เริ่มเตรียมการเทรน mBERT...")
    model_name_bert = "google-bert/bert-base-multilingual-cased"
    bert_tokenizer = AutoTokenizer.from_pretrained(model_name_bert)

    def tokenize_function(examples):
        return bert_tokenizer(examples["text_thai"], padding="max_length", truncation=True, max_length=256)

    raw_dataset = Dataset.from_pandas(df_th[['text_thai', 'label']])
    raw_dataset = raw_dataset.rename_column("label", "labels")
    tokenized_ds = raw_dataset.train_test_split(test_size=0.2, seed=42).map(tokenize_function, batched=True)

    # --- 5. เทรนโมเดล ---
    model = AutoModelForSequenceClassification.from_pretrained(model_name_bert, num_labels=2)

    args = TrainingArguments(
        output_dir="./thai_phishing_results",
        per_device_train_batch_size=8,
        num_train_epochs=2,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["test"],
        compute_metrics=lambda p: {'accuracy': accuracy_score(p.label_ids, p.predictions.argmax(-1))}
    )

    trainer.train()

    # สรุปผล
    eval_res = trainer.evaluate()
    print(f"\n📊 [Final Result] Accuracy: {eval_res['eval_accuracy']:.4f}")
else:
    print("❌ การแปลล้มเหลว ข้อมูลเป็น 0 แถว")

🔄 กำลังโหลดและสุ่มข้อมูล 1,000 แถว...
🚀 กำลังโหลดโมเดล NLLB-200...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


⏳ เริ่มแปลภาษา 1,000 แถวแบบ Batch... (คาดว่าใช้เวลา 10-15 นาที)
✅ แปลเสร็จแล้ว 160/1000 แถว...
✅ แปลเสร็จแล้ว 320/1000 แถว...
✅ แปลเสร็จแล้ว 480/1000 แถว...
✅ แปลเสร็จแล้ว 640/1000 แถว...
✅ แปลเสร็จแล้ว 800/1000 แถว...
✅ แปลเสร็จแล้ว 960/1000 แถว...
✅ แปลเสร็จแล้ว 1000/1000 แถว...
📊 สรุปข้อมูลที่แปลสำเร็จ: 1000 แถว
🔥 เริ่มเตรียมการเทรน mBERT...


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.491265,0.760000
2,No log,0.445742,0.790000



📊 [Final Result] Accuracy: 0.7900
